In [11]:
#run this everytime an edit is made in utils.py so that we can use the helper functions in this Python Notebook
import importlib
import utils
importlib.reload(utils)

<module 'utils' from '/Users/matthewliew/FINM 375-a/matthew-work/hw-1/utils.py'>

In [4]:
import pandas as pd
import numpy as np

#Loading in discount rate data
discount_curve = pd.read_excel("discount_curve_2025-02-13.xlsx")
display(discount_curve.head())

#Loading in callable bonds description
callable_bonds_info = pd.read_excel("callable_bonds_2025-02-18.xlsx", sheet_name="info")
display(callable_bonds_info.head())

#Loading in callable bonds
callable_bonds = pd.read_excel("callable_bonds_2025-02-18.xlsx", sheet_name="quotes")
display(callable_bonds.head())

,ttm,maturity date,spot rate,discount
0,0.5,2025-08-13,0.043743,0.978597
1,1.0,2026-02-13,0.042890,0.958451
2,1.5,2026-08-13,0.042238,0.939228
3,2.0,2027-02-13,0.041843,0.920515
4,2.5,2027-08-13,0.041632,0.902117


,info,FHLMC 4.55 02/11/28
0,CUSIP,3134HA6A6
1,Issuer,FREDDIE MAC
2,Maturity Type,CALLABLE
3,Issuer Industry,GOVT AGENCY
4,Amount Issued,1000000000


,quotes,FHLMC 4.55 02/11/28
0,Date Quoted,2025-02-18 00:00:00
1,TTM,2.976044
2,Clean Price,99.72
3,Dirty Price,99.83375
4,Accrued Interest,0.11375


## 1.1.
Consider a bond with:
- T=3
- face value of N=100
- coupons at annual frequency
- annualized coupon rate of cpn=6%.
Use the bond-pricing formula along with the discount rates in the data file to price this bond.



In [10]:
#Bond Specs
T = 3
FV = 100
coupon_rate = 0.06

vanilla_bond = utils.price_vanilla_bond(maturity=T, face_value=FV, coupon_rate=0.06, discount_curve=discount_curve)
print(vanilla_bond)

104.98670645677134


## 1.2.
Suppose the bond is callable by the issuer.
- European style
- expiration of Topt=1.5
- (clean) strike=100
- vol of 2.68%
- forward price of 103.31.

What is the value of the issuer’s call option?

In [15]:
#Callable Specifications
expiration = 1.5
strike = 100
vol = 0.0268
forward_price = 103.31

call_option_price = utils.call_option_bond(expiration= expiration, 
                                           strike = strike, 
                                           vol = vol, 
                                           forward_price=forward_price
                                           ,discount_curve= discount_curve)
print(call_option_price)

3.373734949447789


## 1.3.
What is the price of the callable bond?

The callable bond is the bond issued with an embedded call option (long the issuer.) Thus, it is the value of the vanilla bond minus the value of the call option.

In [16]:
callable_bond_price = vanilla_bond - call_option_price
print(callable_bond_price)

101.61297150732355


## 1.4.
Which assumptions of Black’s formula do we prefer to Black-Scholes for this problem?

We prefer Black's model because the option is written on a bond whose forward price is more naturally modeled as lognormal. Black assumes the forward price follows GBM under the forward measure, which is more appropriate for interest rate products. In contrast, Black-Scholers assumes the spot follows GBM, which is les srealistic for bonds whose prices depend on the term structure of interest rates.

We prefer Black's model because it assumes a deterministic (effective constant) interest-rate term structure and models the forward price of the bond as lognormal under the forward measure. This avoids the need to model stochastic interest rates. Since the callable bond’s embedded option is naturally an option on the bond’s forward price, Black’s assumptions are more appropriate.


## 1.5
Redo 1.2. Suppose the market prices the call option at 3.50.

Solve for the implied volatility.

In [18]:
from scipy.optimize import brentq
market_price = 3.50

implied_vol = brentq(
    lambda vol: utils.call_option_bond(expiration, strike, vol, forward_price, discount_curve) - market_price,
    0.001,  # lower bound on vol
    2.0     # upper bound on vol
)

print(implied_vol)

0.030946074470700784
